# El Niño Over the Years — Project 3 Checkpoint

## What is El Niño?

Along the equator in the Pacific, the **trade winds** blow from east to west and push warm surface water toward Indonesia and Australia.

* **El Niño:** the trade winds get **weaker**, the warm water drifts back east, and the central and eastern Pacific heat up.
* **La Niña:** the trade winds get **stronger**, and the eastern Pacific gets even colder than usual.

El Niño is measured in the **Niño 3.4 region**, a box of ocean in the central Pacific (5°S–5°N, 170°W–120°W).
The **anomaly** is how much warmer or colder that water is than its average for the same month (we use 1981–2010 as "average").
The **ONI** is that anomaly averaged over 3 months: **+0.5 °C or more for 5+ months = El Niño**, **−0.5 °C or less = La Niña**.

## Our data

CMIP6 **AMIP** experiment from NCAR's **CESM2** climate model, 1950–2014 (Google Cloud `gs://cmip6`).
AMIP runs feed the model the **real, observed ocean temperatures**, so every El Niño happens in the year it really did.
The model then **simulates** how the atmosphere responds, including **rainfall**.
So the ocean temperatures are real, and the rainfall is the model's simulation.

Right now we only have **one simulation run** (`r1i1p1f1`). A single run includes random weather noise,
so we plan to pull more runs (CESM2 has 10) and average the rainfall across them.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

DATA = Path("../data")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

nino = pd.read_csv(DATA / "nino34.csv")
events = pd.read_csv(DATA / "events.csv")
grid = json.load(open(DATA / "grid.json"))

lat = np.array(grid["lat"])
lon = np.array(grid["lon"]) % 360                 # 0–360 so the Pacific sits in the middle of the map
order = np.argsort(lon)
lon = lon[order]
land = np.array(grid["land"]).reshape(len(lat), -1)[:, order]

def load_year(var, year):
    # 12 monthly maps for one year, as (12, lat, lon) in real units
    d = json.load(open(DATA / var / f"{year}.json"))
    a = np.array([[np.nan if v is None else v for v in m] for m in d["months"]], dtype=float) * d["scale"]
    return a.reshape(12, len(lat), -1)[:, :, order]

def draw_land(ax):
    ax.contourf(lon, lat, land, levels=[0.5, 1.01], colors="#d9d6cf")
    ax.contour(lon, lat, land, levels=[0.5], colors="#777", linewidths=0.5)

RED, BLUE, GRAY = "#c0392b", "#2471a3", "#bbbbbb"

## Graph 1 — The ONI timeline

In [ ]:
colors = np.where(nino.phase == "El Niño", RED, np.where(nino.phase == "La Niña", BLUE, GRAY))
x = nino.year + (nino.month - 0.5) / 12

fig, ax = plt.subplots(figsize=(12, 3.6))
ax.bar(x, nino.oni, width=1/12, color=colors)
for y in (0.5, -0.5):
    ax.axhline(y, color="gray", lw=0.6, ls="--")
for label, xx, yy in [("1982–83", 1983.0, 2.75), ("1997–98", 1998.0, 2.75), ("1988–89", 1988.9, -2.5)]:
    ax.annotate(label, (xx, yy), ha="center", fontsize=9)

n_nino = (events.phase == "El Niño").sum()
n_nina = (events.phase == "La Niña").sum()
ax.set(title=f"Oceanic Niño Index, 1950–2014: {n_nino} El Niño and {n_nina} La Niña events",
       ylabel="ONI (°C)", xlim=(1950, 2015), ylim=(-2.8, 3.1))
ax.text(1950.3, -2.65, "Red = El Niño, blue = La Niña, gray = neutral. Dashed lines = ±0.5 °C threshold.",
        fontsize=8, color="#555")
plt.show()

**What it shows:** each bar is one month's ONI, the ocean temperature anomaly in the Niño 3.4 box.
Over 65 years there are 34 events, and their strength varies a lot: 1982–83 and 1997–98 reach about +2.7 °C, while about half of the El Niños never get past +1.5 °C.

## Graph 2 — What one event looks like (Emiliano's Hovmöller)

In [ ]:
hov = json.load(open(DATA / "hovmoller.json"))
hov_values = np.array(hov["values"], dtype=float) * hov["scale"]
dates = pd.to_datetime(hov["dates"])
longitudes = np.array(hov["lon"])

selected = (dates >= "1996-01-01") & (dates < "2001-01-01")
selected_dates = dates[selected]
date_edges = pd.date_range(start=selected_dates[0],
                           end=selected_dates[-1] + pd.offsets.MonthBegin(1), freq="MS")
lon_edges = np.concatenate([
    [longitudes[0] - (longitudes[1] - longitudes[0]) / 2],
    (longitudes[:-1] + longitudes[1:]) / 2,
    [longitudes[-1] + (longitudes[-1] - longitudes[-2]) / 2],
])

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.pcolormesh(lon_edges, mdates.date2num(date_edges), np.ma.masked_invalid(hov_values[selected]),
                   cmap="RdBu_r", vmin=-3, vmax=3, shading="flat", rasterized=True)
ax.invert_yaxis()
ax.yaxis_date()
ax.yaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax.yaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.set_xticks([120, 150, 180, 210, 240, 270])
ax.set_xticklabels(["120°E", "150°E", "180°", "150°W", "120°W", "90°W"])
ax.set(title="How did Pacific temperature anomalies evolve, 1996–2000?",
       xlabel="Longitude: western Pacific → eastern Pacific", ylabel="Month")
fig.colorbar(im, ax=ax, pad=0.03, extend="both").set_label("Sea surface temperature anomaly (°C)")
fig.text(0.5, 0.02, "Averaged across 5°S–5°N • Baseline: 1981–2010 calendar-month means",
         ha="center", fontsize=9, color="#555555")
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

**What it shows:** a Hovmöller diagram. The x-axis runs along the equator from Indonesia (left) to South America (right), time runs down, and color is the temperature anomaly.
You can watch the 1997–98 El Niño spread warm water across the eastern Pacific, then flip straight into a La Niña in 1998–2000.

## Graph 3 — The global footprint: December 1997 vs December 1988

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, (year, label) in zip(axes, [(1997, "December 1997 (El Niño)"), (1988, "December 1988 (La Niña)")]):
    dec = load_year("sst", year)[11]
    im = ax.pcolormesh(lon, lat, dec, cmap="RdBu_r", vmin=-3, vmax=3, shading="nearest")
    draw_land(ax)
    ax.set(title=label, ylim=(-60, 70), xlabel="longitude (°E)")
axes[0].set_ylabel("latitude")
fig.colorbar(im, ax=axes, shrink=0.9, label="Sea surface temperature anomaly (°C)")
fig.suptitle("Ocean temperature anomaly at the peak of an El Niño and a La Niña", x=0.45, y=1.02)
plt.show()

**What it shows:** ocean temperature anomaly maps (this is the same ~4°×5° grid our D3 map will use).
In December 1997 a long tongue of warm water stretches along the equator to South America; in December 1988 the same region is cold.
El Niño and La Niña are mirror images.

## Graph 4 — Rain shifts worldwide

In [ ]:
pr = np.concatenate([load_year("pr", y) for y in range(1950, 2015)])      # (780, lat, lon), mm/day anomaly
winter = nino.month.isin([12, 1, 2]).values

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, phase in zip(axes, ["El Niño", "La Niña"]):
    months = winter & (nino.phase.values == phase)
    im = ax.pcolormesh(lon, lat, pr[months].mean(axis=0), cmap="BrBG", vmin=-4, vmax=4, shading="nearest")
    ax.contour(lon, lat, land, levels=[0.5], colors="#555", linewidths=0.6)
    ax.set(title=f"{phase} winters ({months.sum()} months)", ylim=(-60, 70), xlabel="longitude (°E)")
axes[0].set_ylabel("latitude")
fig.colorbar(im, ax=axes, shrink=0.9, label="Rainfall anomaly (mm/day)")
fig.suptitle("Simulated December–February rainfall anomaly", x=0.45, y=1.02)
plt.show()

**What it shows:** average winter (Dec–Feb) rainfall anomaly during El Niño vs La Niña. Green = wetter than normal, brown = drier.
During El Niño the rain follows the warm water to the central Pacific, while Indonesia, Australia and northern South America dry out. La Niña flips the pattern.

## Graph 5 — Does El Niño bring rain to California?

In [ ]:
LA, LO = np.meshgrid(lat, lon, indexing="ij")
ca_cells = (LA >= 32) & (LA <= 42) & (LO >= 236) & (LO <= 246) & (land >= 0.5)   # 124°W–114°W, land only
ca = pr[:, ca_cells].mean(axis=1)

df = pd.DataFrame({"winter": nino.year + (nino.month == 12), "oni": nino.oni, "ca": ca})[winter]
g = df.groupby("winter")
win = g.mean()[g.size() == 3].dropna()          # complete Dec–Jan–Feb winters only
r = np.corrcoef(win.oni, win.ca)[0, 1]

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(win.oni, win.ca, c=win.oni, cmap="RdBu_r", vmin=-2, vmax=2, edgecolor="k", lw=0.4, s=45)
for yr, row in win[win.oni.abs() > 1.5].iterrows():
    ax.annotate(f"{yr - 1}–{str(yr)[2:]}", (row.oni, row.ca), fontsize=8, xytext=(4, 2), textcoords="offset points")
ax.axhline(0, color="gray", lw=0.6)
ax.axvline(0, color="gray", lw=0.6)
ax.set(title=f"Winter ONI vs California rainfall anomaly (r = {r:.2f})",
       xlabel="Dec–Feb average ONI (°C)", ylabel="Dec–Feb rainfall anomaly (mm/day)")
plt.show()

**What it shows:** each dot is one winter. The two biggest El Niños (1982–83, 1997–98) soaked California, but the strong 1991–92 El Niño left it drier than normal.
The overall link is weak (r ≈ 0.3): a strong El Niño makes a wet California winter more likely, but it isn't guaranteed.

## Graph 6 — The storms follow the warm water (David's Graph B)

In [ ]:
import html as html_lib, urllib.request
from IPython.display import HTML, display

page = Path("out/graphB_storm_clouds.html").read_text(encoding="utf-8")
cdn = '<script src="https://cdn.jsdelivr.net/npm/d3@7.9.0/dist/d3.min.js"></script>'
try:   # inline D3 so the chart also shows inside VS Code
    d3_src = urllib.request.urlopen("https://cdn.jsdelivr.net/npm/d3@7.9.0/dist/d3.min.js", timeout=15).read().decode()
    page = page.replace(cdn, f"<script>{d3_src}</script>")
except Exception:
    pass
import warnings
warnings.filterwarnings("ignore", message="Consider using IPython.display.IFrame")
display(HTML(f'<iframe srcdoc="{html_lib.escape(page, quote=True)}" '
             f'style="width:100%;max-width:920px;height:1030px;border:0"></iframe>'))
print("Full size: open notebooks/out/graphB_storm_clouds.html in a browser.")

**What it shows:** storm clouds along the equator, measured by the heat escaping to space (cold storm tops = low values).
Teal = stormier than normal, brown = clearer. The black line marks how far east the storms reach.
Drag the top strip to pick years: during 1997–98 the storms march east across the Pacific with the warm water, and they pull back west during La Niña.
This is the link between the ocean (Graphs 1–3) and the rain (Graphs 4–5).

## Our question and what we're building next

**Question:** *Not all El Niños are equal: how does an El Niño's strength change where the world gets warmer water and more (or less) rain?*

**Final interactive visualization (D3, GitHub Pages):**

* **Top: a timeline slider** of the Niño 3.4 anomaly (ONI) for every month from 1950–2014, colored by El Niño / La Niña, similar to David's year strip. Dragging or clicking it picks a month or a range of months.
* **Bottom: a world map** for the selected time, with a **toggle between ocean temperature and rainfall**. Moving the slider through an event like 1997–98 shows the warm water and the rain shifting together.
* **Details on demand:** hover any map cell to see its value; hover the timeline to see the month's ONI and phase.

**Next steps:**
1. Pull more CESM2 AMIP runs and average the rainfall to cut down on random noise.
2. Build the D3 timeline + map using the files already in `data/` (`nino34.csv`, `grid.json`, `sst/`, `pr/`).
3. Add the interactions (slider, toggle, tooltips), then deploy on GitHub Pages.
4. Write up our design choices, data transformations (anomalies, 3-month averaging, coarsening the grid) and development process.